In [39]:
import numpy as np
from scipy.stats import t, norm
from copulae.elliptical import GaussianCopula
from copulae.elliptical import StudentCopula
import yfinance as yf
import matplotlib.pyplot as plt
import ruptures as rpt
import pandas as pd

In [2]:
yahoo_tickers = {
    # Sectors / Industries
    "Technology": "XLK",
    "Financials": "XLF",
    "Energy": "XLE",
    "Healthcare": "XLV",
    "Industrials": "XLI",
    "Utilities": "XLU",
    "Consumer Discretionary": "XLY",
    "Consumer Staples": "XLP",
    "Materials": "XLB",
    "Real Estate": "RWR", # XLRE started in 2016
    "Communication Services": "VOX", # XLC started in 2016

    # Commodities
    "Broad Commodities": "DBC",
    "Crude Oil": "USO",
    "Gold": "GLD",
    # "Silver": "SLV",
    "Natural Gas": "UNG",
    "Agriculture": "DBA",

    # FX / Currencies
    "US Dollar Index": "UUP",
    "EUR/USD": "FXE",
    # "GBP/USD": "FXB",
    "JPY/USD": "FXY",
    # "CAD/USD": "FXC"
}

In [19]:
# get yfinance data
tickers = list(yahoo_tickers.values())
data = yf.download(tickers = tickers, start="2008-01-01", end="2025-01-01", interval="1d") #, period="5y",

/var/folders/1d/dkqlxg7d3yv8ccfw3vkrtnsm0000gn/T/ipykernel_1408/3052108859.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers = tickers, start="2008-01-01", end="2025-01-01", interval="1d") #, period="5y",
[*********************100%***********************]  19 of 19 completed


In [22]:
def compute_returns_from_yahoo_data(data):
    returns = data['Close'].pct_change().dropna()
    return returns

In [25]:
returns = compute_returns_from_yahoo_data(data)
returns_np = returns.to_numpy()

In [55]:
def get_breakpoints(returns, n_pts=10, model="normal", min_penalty = 10, max_penalty = 1000, tolerance=0):
    algo = rpt.Pelt(model=model).fit(returns) # rbf

    pred_bkpts = []
    pen = max_penalty
    while not pred_bkpts or abs(len(pred_bkpts) - n_pts) > tolerance:
        if len(pred_bkpts) > n_pts:
            min_penalty = pen
            pen = (pen + max_penalty) // 2
        else:
            max_penalty = pen
            pen = (pen + min_penalty) // 2
        print("Trying Pen", pen)
        *pred_bkpts, num_samples = algo.predict(pen=pen)
        print("Num bkpts", len(pred_bkpts))

        if pen == max_penalty or pen == min_penalty:
            break

    return pred_bkpts, pen
        

In [62]:
# pred_bkpts, pen = get_breakpoints(returns_np[:, 0], n_pts=5, max_penalty=200)

In [60]:
def plot_breakpoints(price_data: pd.Series, pred_bkpts: list):
    plt.plot(price_data)
    plt.vlines(price_data.index[pred_bkpts], ymin=min(price_data), ymax=max(price_data), colors=["black"]*len(pred_bkpts), linestyles=["--"]*len(pred_bkpts))
    plt.show()

In [63]:
# plot_breakpoints(data.iloc[:, 0], pred_bkpts)

In [ ]:
def get_max_corr(mat):
    mask = ~np.eye(mat.shape[0], dtype=bool)
    off_diag_indices = np.argwhere(mask)
    max_idx = mat[mask].argmax()
    i, j = off_diag_indices[max_idx]
    return (i, j), mat[i, j]

In [ ]:
def construct_corr_from_rhos(dim, rho_array):
    upper_triangular = np.zeros((dim, dim))
    indices = np.triu_indices(dim, k=1)
    upper_triangular[indices] = rho_array
    corr = upper_triangular + upper_triangular.T + np.eye(dim)

    return corr

In [ ]:
def compute_tail_dependence(df, corr):

    def t_tail_dependence(df, rho):
        # df = degrees of freedom
        # rho = correlation between variables i,j
        arg = -np.sqrt((df + 1) * (1 - rho) / (1 + rho))
        return 2 * t.cdf(arg, df + 1)

    # Example for all pairs in a fitted copula:
    tail_dep_matrix = np.zeros_like(corr)

    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            tail_dep_matrix[i, j] = t_tail_dependence(df, corr[i, j])
        
    return tail_dep_matrix

In [ ]:
g_copula = GaussianCopula(dim=19)
t_copula = StudentCopula(dim=19)
# t_copula.fit(U, to_pobs=False, method="ml", verbose=1000)


# g_copula.fit(U, to_pobs=False, method="ml", verbose=1000)
# rho_mat = g_copula.params
# g_corr = construct_corr_from_rhos(dim=19, rho_array=rho_mat)

In [10]:
g_copula.name, t_copula.name

('Gaussian', 'Student')

In [ ]:
def transform_data_to_uniform(data, dist = "t"):
    '''
        Rows are assumed to be entries and columns are variables
    '''

    if dist not in ["t", "norm"]:
        raise ValueError("distribution must be normal or t")

    if dist == "t":
        model = t
    elif dist == "norm":
        model = norm
    
    params_by_col = {}
    Uniform_cols = []
    for i in range(data.shape[1]):
        params = model.fit(data[:, i])
        
        if dist == "t":
            U_col = model.cdf(data[:, i], loc=params[1], scale=params[2], df=params[0])
        elif dist == "norm":
            U_col = model.cdf(data[:, i], loc=params[0], scale=params[1])
        
        Uniform_cols.append(U_col)
        params_by_col[i] = params

    U = np.array(Uniform_cols).T

    return U, params

In [ ]:
def transform_uniform_to_marginal(U: np.array, rv_params: dict, dist: str = "t"):
    '''
        Rows are assumed to be entries and columns are variables
    '''

    if dist not in ["t", "norm"]:
        raise ValueError("distribution must be normal or t")

    if dist == "t":
        model = t
    elif dist == "norm":
        model = norm
    
    data_cols = []
    for i in range(U.shape[1]):
        params = rv_params[i]
        if dist == "t":
            data_col = model.ppf(U[:, i], loc=params[1], scale=params[2], df=params[0])
        elif dist == "norm":
            data_col = model.ppf(U[:, i], loc=params[0], scale=params[1])
        
        data_cols.append(data_col)

    data = np.array(data_cols).T
    return data

In [ ]:
def simulate_from_copula(copula: GaussianCopula | StudentCopula, n_sims):
    sim_U = copula.random(n_sims)
    return sim_U

In [71]:
returns

Ticker,DBA,DBC,FXE,FXY,GLD,RWR,UNG,USO,UUP,VOX,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY
Date,,,,,,,,,,,,,,,,,,,
2008-01-03,0.023631,0.007709,0.002716,0.002303,0.008367,-0.029327,-0.024314,-0.001274,0.002129,-0.002694,0.018160,0.011824,-0.006347,0.000779,0.001532,-0.005283,-0.001426,0.006010,-0.011181
2008-01-04,0.010520,-0.005814,0.000542,0.007768,-0.005142,-0.042177,0.011522,-0.013526,-0.010620,-0.024034,-0.030202,-0.036549,-0.028389,-0.020498,-0.038991,-0.004250,0.007614,-0.010242,-0.031406
2008-01-07,-0.013303,-0.018159,-0.003925,-0.006514,-0.004229,0.006714,0.007947,-0.023412,0.009876,0.010791,-0.013732,-0.003871,0.002557,-0.012451,-0.008751,0.011735,0.020779,0.019833,0.003567
2008-01-08,0.011137,0.009091,0.000951,0.000874,0.023711,-0.032573,0.009198,0.007417,-0.001275,-0.035176,-0.015415,-0.017487,-0.036430,-0.022263,-0.026485,-0.007381,0.000231,0.008174,-0.019386
2008-01-09,-0.004058,-0.009941,-0.002783,-0.008079,-0.002650,0.018118,0.022135,-0.010649,0.007663,0.003688,0.006313,0.013184,0.018903,0.003018,0.014015,0.013456,0.010869,0.017892,0.007248
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,0.005305,0.005736,-0.001561,-0.001362,0.001992,0.008953,0.032630,0.008076,0.002731,0.009699,0.005407,0.008459,0.011772,0.007813,0.010333,0.006589,0.005522,0.004108,0.023127
2024-12-26,-0.010554,-0.002852,0.003232,-0.004602,0.006751,0.002218,-0.047708,-0.007060,-0.002043,-0.001256,-0.001520,-0.000827,0.002449,0.000745,0.000665,0.003147,-0.002354,0.002010,-0.003710
2024-12-27,-0.015238,0.004766,0.000520,0.001027,-0.006871,-0.010061,0.025374,0.009845,-0.001706,-0.011252,-0.005386,-0.000118,-0.007331,-0.007448,-0.013295,-0.004894,-0.002883,-0.004656,-0.016524


In [73]:
sofr_rate = pd.read_excel("sofr_historical.xlsx")

/Users/willneuner/Desktop/FINTECH545/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
sofr_rate = pd.read_excel("sofr_historical.xlsx")
sofr_rate['Date'] = pd.to_datetime(sofr_rate["Effective Date"])
sofr_rate_series = sofr_rate.set_index("Date")["Rate (%)"]

In [ ]:
def compute_portfolio_es():

    

In [ ]:
def compute_max_es_sharpe_weights(means, cov, rfr, weight_bounds = (0, None)):
    
    # minimize SSE CSD function
    def objective_function(w, means, cov, rfr):
        vol = np.sqrt(w.T.dot(cov).dot(w))
        sharpe = (w.dot(means) - rfr) / vol
        return sharpe

    # Equality constraint: sum(w) = 1 -> sum(w) -1 = 0
    constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    # bounds
    bounds = (weight_bounds,)*cov.shape[0]

    # initial guess
    w0 = (np.ones((cov.shape[0], 1)) / cov.shape[0]).ravel()
    result = minimize(lambda x: -objective_function(x, means, cov, rfr), w0, method='SLSQP', bounds=bounds, constraints=[constraint], options={'ftol': 1e-14, 'maxiter': 1000, 'disp': True})

    return result.x